# Vehicle Destination Lab — end-to-end notebook

This notebook calls the reusable functions in `vehicle_destination/`; it does not duplicate pipeline logic. It executes raw-data cleaning, trip reconstruction, leakage-safe sample preparation, baseline training, evaluation, inference, and an interactive predicted-versus-actual map. Run the cells in order.

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'configs' / 'default.yaml').exists() and (candidate / 'vehicle_destination').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook from inside the vehicle_destination_lab project.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import HTML, display
import pandas as pd
from vehicle_destination.config import load_config
from vehicle_destination.dataset import prepare_samples
from vehicle_destination.inference import VinInferenceRequest
from vehicle_destination.pipeline import (
    build_trip_artifacts, evaluate_saved_predictions, predict_vin_scenario, train_model
)
from vehicle_destination.ui.maps import build_leaflet_html, build_map_payload

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'default.yaml'
config = load_config(CONFIG_PATH)
NOTEBOOK_RUN = PROJECT_ROOT / 'artifacts' / 'notebook_run'
NOTEBOOK_RUN.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')
print(f'Configuration hash: {config.reproducibility_hash[:16]}…')
print(f'Notebook outputs: {NOTEBOOK_RUN}')

## 1. Inspect raw telemetry

In [ ]:
raw_path = config.path(config.data.raw_data)
raw = pd.read_csv(raw_path)
display(pd.DataFrame({
    'metric': ['raw rows', 'VINs', 'triggers', 'signals'],
    'value': [len(raw), raw['vin'].nunique(), raw['triggerOrContext'].nunique(), raw['name'].nunique()]
}))
display(raw.head())

## 2. Clean raw rows and reconstruct trips
The source function preserves the raw file and writes cleaning/rejection audits alongside the accepted trips.

In [ ]:
trip_stage = build_trip_artifacts(config, raw_path=raw_path, output_dir=NOTEBOOK_RUN)
display(pd.DataFrame([trip_stage.summary]).T.rename(columns={0: 'value'}))
display(pd.DataFrame(trip_stage.outputs.items(), columns=['artifact', 'path']))

trips = pd.read_csv(trip_stage.outputs['trips'])
points = pd.read_csv(trip_stage.outputs['points'])
cleaning_audit = pd.read_csv(trip_stage.outputs['cleaning_audit'])
display(cleaning_audit)
display(trips.head())
print(f'{len(trips)} accepted trips; {len(points)} reconstructed points; {trips.VIN.nunique()} VINs')

## 3. Build leakage-safe model samples and inspect VIN splits

In [ ]:
samples, split_manifest = prepare_samples(trips, config.dataset)
split_summary = samples.groupby('split').agg(samples=('sample_id', 'size'), VINs=('VIN', 'nunique'), trips=('trip_id', 'nunique'))
display(split_summary)
assert set(split_manifest.train_vins).isdisjoint(split_manifest.validation_vins)
assert set(split_manifest.train_vins).isdisjoint(split_manifest.test_vins)
assert set(split_manifest.validation_vins).isdisjoint(split_manifest.test_vins)
print('VIN-disjoint leakage check: PASSED')
display(samples[['sample_id', 'vehicle_id', 'trip_id', 'split', 'prefix_fraction', 'history_count', 'actual_cell']].head(10))

## 4. Train the baseline from source code
This writes a model, metadata, test predictions, grouped metrics, and the resolved configuration into `artifacts/notebook_run/models/baseline`.

In [ ]:
model_dir = NOTEBOOK_RUN / 'models' / 'baseline'
training = train_model(config, engine='baseline', trips_path=trip_stage.outputs['trips'], output_dir=model_dir)
display(pd.DataFrame([training.summary]).T.rename(columns={0: 'test value'}))
display(pd.DataFrame(training.outputs.items(), columns=['artifact', 'path']))

## 5. Evaluate saved predictions

In [ ]:
metrics = evaluate_saved_predictions(training.outputs['predictions'], config, split='test')
display(pd.DataFrame([metrics]).T.rename(columns={0: 'value'}))
predictions = pd.read_csv(training.outputs['predictions'])
display(predictions.query("split == 'test'")[['trip_id', 'prefix_fraction', 'predicted_cell_probability', 'error_km', 'actual_rank']].head(10))

## 6. Run VIN-driven inference with editable inputs
Choose a VIN and reference trip, then override any information that is available at prediction time. The actual destination is retained only for comparison and never enters the feature vector.

In [ ]:
selected_trip_id = predictions.query("split == 'test'")['trip_id'].iloc[0]
selected_vin = trips.loc[trips.trip_id.eq(selected_trip_id), 'VIN'].iloc[0]
request = VinInferenceRequest(
    vin=selected_vin,
    reference_trip_id=selected_trip_id,
    departure_time=None,        # set an ISO UTC timestamp to override
    origin_latitude=None,       # set latitude and longitude together
    origin_longitude=None,
    prefix_fraction=0.25,
    prefix_points=None,         # or tuple((lat, lon), ...)
    history_trip_ids=None,      # automatic eligible completed history
    top_k=5,
)
scenario = predict_vin_scenario(
    config, request=request, engine='baseline',
    trips_path=trip_stage.outputs['trips'], model_dir=model_dir
)
prediction_frame = scenario.prediction
selected_trip = scenario.reference_trip
model = scenario.model
prediction = prediction_frame.iloc[0]
display(pd.DataFrame([request.to_dict()]).T.rename(columns={0: 'resolved input'}))
display(pd.DataFrame(scenario.provenance['input_sources'].items(), columns=['input', 'source']))
display(prediction_frame[['trip_id', 'prefix_fraction', 'predicted_latitude', 'predicted_longitude', 'actual_latitude', 'actual_longitude', 'error_km']])
candidates = pd.DataFrame(json.loads(prediction['top_k_candidates']))
display(candidates)

## 7. Visualize trajectory, predictions, and actual destination on the map

In [ ]:
payload = build_map_payload(
    selected_trip, prediction, model.grid,
    observed_prefix=scenario.sample.iloc[0]['trajectory_prefix']
)
visible_layers = {'Full trajectory', 'Observed prefix', 'Candidates', 'Destinations', 'Candidate cells', 'Connection lines'}
map_html = build_leaflet_html(payload, visible_layers)
map_path = NOTEBOOK_RUN / 'prediction_map.html'
map_path.write_text(map_html, encoding='utf-8')
display(HTML(f'<iframe src="{map_path.as_uri()}" width="100%" height="620" style="border:1px solid #d7dde5;border-radius:8px"></iframe>'))
print(f'Interactive map saved to: {map_path}')

## Optional: train the TensorFlow/Keras model
Install `pip install -e ".[tensorflow]"` and the Graphviz system package, then explicitly change `RUN_KERAS` to `True`. The bundled 70-trip fixture validates execution only; it is too small for production performance claims.

In [ ]:
RUN_KERAS = False
if RUN_KERAS:
    keras_training = train_model(
        config, engine='keras', trips_path=trip_stage.outputs['trips'],
        output_dir=NOTEBOOK_RUN / 'models' / 'keras'
    )
    model = keras_training.model
    assert model is not None
    model.summary()
    model_plot = model.plot(
        NOTEBOOK_RUN / 'models' / 'keras' / 'model_architecture.png'
    )
    display(model_plot)
    display(pd.DataFrame([keras_training.summary]).T.rename(columns={0: 'test value'}))
else:
    print('Keras training skipped. Set RUN_KERAS = True after installing the TensorFlow extra.')

## Outputs
All generated files are under `artifacts/notebook_run/`. Re-running the training cells replaces only that notebook-specific run. The original raw telemetry and bundled demo artifacts remain unchanged.